
### Partea A – Pregătirea datelor

**A1.** Citiți `Mt_DNA_tari.csv` cu `Code` ca index. Câte valori lipsă are fiecare coloană?
Afișați doar coloanele cu valori lipsă.

**A2.** Înlocuiți valorile lipsă cu `nan_replace_df`. Verificați că nu mai rămâne nicio valoare `NaN`.

**A3.** Construiți lista `variabile_observate` (haplogrupurile, fără `Country`) și matricea `x`.
Precizați `n` (numărul de instanțe) și `m` (numărul de variabile).

In [11]:
import pandas as pd
import numpy as np
import utils as utils

dna_df = pd.read_csv("./data_in/ADN/Mt_DNA_tari.csv", index_col="Code")
numerical_variables = dna_df.columns[1:]
dna_df = utils.nan_replace_df(dna_df[numerical_variables])

dna_df.head()

,L,HV,H,H1_H3,H5,HV0_V,J,T1,T2,U2,U3,U4,U5,U,K,Eu,W,X,Alt
Code,,,,,,,,,,,,,,,,,,,
ALB,0.3,0.9,47.9,3.00,4.0,3.4,7.1,4.3,5.5,1.9,0.6,2.5,6.4,2.8,5.2,1.8,2.5,1.8,5.1
AUT,0.0,0.8,44.9,16.50,3.7,1.9,8.8,4.0,8.0,1.9,0.8,4.6,8.6,1.4,8.6,1.1,1.3,1.3,2.0
BLR,0.0,2.0,39.1,11.00,2.9,6.0,8.9,2.3,5.1,2.3,2.0,3.1,9.1,6.6,2.3,1.7,3.7,0.3,5.4
BEL,0.0,0.7,46.9,26.80,6.0,2.7,6.0,2.0,9.4,0.7,1.3,2.7,3.4,5.4,12.1,2.0,3.3,1.3,0.0
BIH,0.5,1.1,45.8,16.81,5.6,4.7,8.3,1.8,4.4,0.8,0.8,4.9,8.8,1.1,4.9,3.2,2.9,0.8,5.0



---

### Partea B – Modelul ACP

**B1.** Aplicați `acp(x)` și salvați matricea de corelație în `R.csv`. Trasați corelograma.
Care este perechea de haplogrupuri cu cea mai puternică corelație (în valoare absolută, fără diagonală)?

**B2.** Verificați numeric proprietățile modelului:
- suma valorilor proprii `alpha` este egală cu `m` (datele sunt standardizate);
- vectorii proprii sunt ortonormați: `a.T @ a ≈ I`;
- componentele `c = x_ @ a` sunt necorelate, iar varianța componentei `j` este `alpha[j]`.

Folosiți `np.allclose` pentru fiecare verificare.

**B3.** *(Găsiți problema)* Afișați `alpha.dtype`. Pe acest set de date, `acp` (care folosește
`np.linalg.eig`) întoarce valori proprii **complexe** (`+0.000j`), iar mai târziu operații precum
`nlargest` din pandas vor da eroare. Matricea de corelație este simetrică. Rescrieți funcția ca
`acp_eigh` folosind `np.linalg.eigh` și explicați de ce varianta aceasta este mai potrivită.
Atenție: `eigh` întoarce valorile proprii în ordine **crescătoare**. Folosiți `acp_eigh` în
restul exercițiilor.

**B4.** Rulați ACP pe datele **nestandardizate** (`acp_eigh(x, scal=False)`). Comparați procentul de
varianță explicat de prima componentă în cele două cazuri. Ce variabilă domină componenta 1
în cazul nestandardizat și de ce? *(Indiciu: uitați-vă la abaterile standard ale variabilelor.)*


In [58]:
dna_df = (dna_df-dna_df.mean())
dna_df = dna_df/dna_df.std(ddof=0)

corr, eigenvalues, eigenvectors = utils.acp(dna_df)

corr.to_csv('./data_out/corr.csv')
without_diag = corr - np.eye(corr.shape[0])
without_diag.round(2)
without_diag=without_diag.abs()
without_diag.round(2)
without_diag.max().max()
without_diag.max().idxmax()

long = without_diag.stack().reset_index()
long.columns = ['varX', 'varY', 'value']

wide = long.pivot(index='varX', columns='varY', values='value')

wide

varY,Alt,Eu,H,H1_H3,H5,HV,HV0_V,J,K,L,T1,T2,U,U2,U3,U4,U5,W,X
varX,,,,,,,,,,,,,,,,,,,
Alt,0.000000,0.024922,6.072476e-01,4.399952e-01,0.036553,4.183627e-01,2.404153e-01,0.225050,0.002419,6.341358e-02,0.314187,3.577019e-01,0.264069,1.777256e-01,5.179762e-01,7.877471e-02,0.184866,0.005185,0.372391
Eu,0.024922,0.000000,2.787406e-01,1.463911e-01,0.077396,9.965840e-02,4.137832e-02,0.144879,0.018964,9.239389e-02,0.396949,2.276161e-01,0.181243,9.848484e-02,2.117105e-02,1.936508e-01,0.330517,0.265241,0.230542
H,0.607248,0.278741,4.440892e-16,2.480543e-01,0.017226,5.405003e-01,1.930283e-01,0.165852,0.303958,6.158231e-02,0.075813,1.738410e-01,0.303542,2.207816e-01,4.621413e-01,2.030322e-02,0.027290,0.392168,0.266595
H1_H3,0.439995,0.146391,2.480543e-01,2.220446e-16,0.283597,3.912926e-01,1.609855e-01,0.079696,0.206981,3.186315e-01,0.425897,1.233870e-02,0.008124,2.087226e-01,1.759769e-01,1.932050e-01,0.160532,0.110292,0.232677
H5,0.036553,0.077396,1.722646e-02,2.835971e-01,0.000000,9.046646e-02,2.504502e-01,0.075176,0.094570,2.567569e-01,0.025149,1.320509e-01,0.043495,9.316599e-02,1.783138e-01,3.490969e-01,0.139556,0.075297,0.009340
HV,0.418363,0.099658,5.405003e-01,3.912926e-01,0.090466,2.220446e-16,3.640108e-01,0.039023,0.062507,1.738654e-01,0.000249,5.984032e-02,0.057175,2.344779e-02,3.409635e-01,9.343538e-02,0.215854,0.151691,0.189435
HV0_V,0.240415,0.041378,1.930283e-01,1.609855e-01,0.250450,3.640108e-01,2.220446e-16,0.052691,0.253992,5.397743e-02,0.022644,1.487817e-01,0.282067,7.154858e-02,5.057056e-01,1.435201e-01,0.397885,0.297008,0.359540
J,0.225050,0.144879,1.658524e-01,7.969631e-02,0.075176,3.902283e-02,5.269103e-02,0.000000,0.001120,2.719090e-01,0.320969,4.395492e-01,0.451419,3.615800e-02,3.824712e-01,1.129694e-01,0.047373,0.389363,0.160245
K,0.002419,0.018964,3.039581e-01,2.069812e-01,0.094570,6.250697e-02,2.539922e-01,0.001120,0.000000,2.277099e-01,0.274930,1.576974e-01,0.389984,2.775837e-01,3.437139e-01,2.352571e-01,0.606250,0.085440,0.200252



---

### Partea C – Varianța componentelor

**C1.** Construiți tabelul varianței și salvați-l în `Varianta.csv`.

**C2.** Trasați scree-plot-ul cu `plot_variance`. Câte componente rețin criteriul Kaiser, criteriul
procentului minimal de 80% și criteriul Cattell?

**C3.** Scrieți o funcție `nr_componente(alpha, procent_minimal=80)` care întoarce cele trei
valori **fără** să deseneze nimic. Testați-o pentru pragurile 70%, 80% și 90%.

**C4.** Pornind de la `main.py`, rețineți `nr_min_comp` = minimul dintre criteriile care au dat
un rezultat. De ce este nevoie să ignorăm valorile `None`?

---

### Partea D – Corelații factoriale și scoruri

**D1.** Calculați componentele `c`, scorurile `s = c / sqrt(alpha)` și corelațiile factoriale
`r_xc`. Salvați `C.csv`, `S.csv`, `R_xc.csv`.

**D2.** Arătați că, pentru date standardizate, corelațiile factoriale se pot calcula și direct:
`r_xc = a * sqrt(alpha)`. Verificați cu `np.allclose`.

**D3.** Verificați că fiecare coloană din `S` are media 0 și abaterea standard 1.

**D4.** Trasați cercul corelațiilor pentru (C1, C2). Care haplogrupuri sunt bine reprezentate
(în afara cercului de rază 0.7)? Interpretați axa C1: ce haplogrupuri o definesc în sens pozitiv
și negativ?

**D5.** Trasați plotul scorurilor (C1, C2). Ce grupări de țări apar? Unde se poziționează România?

---

### Partea E – Indicatori de calitate

**E1.** Calculați cosinusurile pătratice. Verificați că suma pe fiecare linie este 1.
Ce țară este cel mai bine reprezentată pe C1? Dar pe planul (C1, C2)?

**E2.** Calculați contribuțiile instanțelor. Verificați că suma pe fiecare coloană este 1.
Afișați primele 3 țări care contribuie cel mai mult la varianța C1 și C2.

**E3.** Calculați comunalitățile. Verificați că, pe ultima coloană, toate comunalitățile sunt 1.
Ce haplogrup este cel mai slab explicat de primele `nr_min_comp` componente?

**E4.** Trasați corelogramele pentru cosinusuri, contribuții și comunalități (ca în `main.py`).

---

### Partea F – Harta scorurilor

**F1.** Citiți `Europa/Europa.shp` cu `geopandas`. Ce coloană folosim pentru legătura cu datele?

**F2.** Faceți `merge` între hartă și `t_c` după `Code`. Câte țări din hartă nu au date?

**F3.** Scrieți în `grafice.py` o funcție `harta(gdf, camp_legatura, t, coloana, titlu)` care
desenează o hartă choroplet pentru o componentă (țările fără date apar în gri). Trasați hărțile
pentru primele `nr_min_comp` componente și salvați-le în `graphics/`.

---

### Partea G – Extensii (opțional)

**G1.** Repetați analiza pe `Mt_DNA_Regiuni.csv`. Atenție: ultima coloană (`Mărime`, dimensiunea
eșantionului) **nu** este un haplogrup și trebuie exclusă. Fișierul nu este UTF-8, deci citiți-l cu
`encoding="latin1"`.

**G2.** Comparați rezultatele mtDNA (linia maternă) cu cele Y-DNA din `main.py` (linia paternă):
în care dintre ele se separă mai clar țările pe C1? Comparați procentul explicat de C1 și numărul
de componente reținute.

**G3.** Implementați ACP prin descompunerea în valori singulare (`np.linalg.svd` aplicat pe `x_`)
și arătați că obțineți aceleași valori proprii. De ce componentele pot avea semnul inversat?